# Reintegración de las funciones de Melanie

In [1]:
# ==========================================================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Importación de librerías y reconstrucción de los DataFrames
#              homologados de la Sección B (incisos 6 y 7).
# Insumo:      src/homologacion.py, data/processed/tabla_homologacion_marcas.csv
# Resultado:   DataFrames cd, uci y tabla_homologacion_marcas.
# ==========================================================================================
# Cada notebook tiene su propia memoria. Las variables cd y uci que creó Melanie 
# existen solo dentro de su notebook mientras corre. Por ser un notebook nuevo repetimos 
# sus mismos pasos para tenerlas en memoria aquí.
# ===========================================================================================
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("src")
import homologacion as hm

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
os.makedirs("figuras", exist_ok=True)

# 1) Carga de datos crudos
cd_raw = hm.cargar_cardekho()
uci_raw = hm.cargar_uci()

# 2) Renombrado a español
cd = cd_raw.rename(columns=hm.RENOMBRAR_CARDEKHO)
uci = uci_raw.rename(columns=hm.RENOMBRAR_UCI)

# 3) Año (para los filtros de la sección C)
cd["Año"] = hm.crear_anio(cd["Edad_Vehiculo"])

# 4) Marca
cd["Marca"] = hm.extraer_marca(cd["Nombre"])
uci["Marca"] = uci["Fabricante"]

# 5) Marca homologada
cd["Marca_Homologada"] = hm.homologar_marca(cd["Marca"])
uci["Marca_Homologada"] = hm.homologar_marca(uci["Marca"])

# 6) Tabla de homologación generada por Melanie
tabla_homologacion_marcas = pd.read_csv("data/processed/tabla_homologacion_marcas.csv", encoding="utf-8-sig")

print("CarDekho:", cd.shape)
print("UCI     :", uci.shape)

CarDekho: (15411, 17)
UCI     : (205, 28)


# Sección B.9  Integración de la información de CarDekho y UCI Automobile

In [2]:
# ==========================================================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Antes de hacer el join, hay que ver cuántas veces se repite cada marca en UCI.
# Insumo:      Tabla de homologacion UCI
# Resultado:   Cuántas filas hay de cada marca.
# ==========================================================================================
uci["Marca_Homologada"].value_counts()

Marca_Homologada
toyota           32
nissan           18
mazda            17
honda            13
mitsubishi       13
subaru           12
volkswagen       12
peugeot          11
volvo            11
dodge             9
bmw               8
mercedes benz     8
audi              7
plymouth          7
saab              6
porsche           5
isuzu             4
alfa romeo        3
chevrolet         3
jaguar            3
renault           2
mercury           1
Name: count, dtype: Int64

In [3]:
# ==========================================================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Conteo de cuántos autos Toyota hay en CarDekho.
# Insumo:      src/homologacion.py, data/processed/tabla_homologacion_marcas.csv
# Resultado:   Valor numérico de la cantidad de autos de Toyota hay en CarDekho
# ==========================================================================================
(cd["Marca_Homologada"] == "toyota").sum()

np.int64(793)

In [4]:
# ===========================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Integración directa (left join) de CarDekho con UCI usando
#              Marca_Homologada como clave, para medir el efecto de una
#              relación muchos a muchos.
# Insumo:      cd, uci
# Resultado:   DataFrame integrado_directo.
# ===========================================================
integrado_directo = pd.merge(cd, uci, on="Marca_Homologada", how="left", suffixes=("", "_UCI"))

print("Filas antes  (CarDekho):", cd.shape[0])
print("Filas después (join)   :", integrado_directo.shape[0])

Filas antes  (CarDekho): 15411
Filas después (join)   : 72380


In [5]:
# ==========================================================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Comprobar que son duplicados de los mismos autos
# Insumo:      integrado_directo
# Resultado:   Valor numércio sobre las repeticiones hay (duplicados)
# ==========================================================================================
integrado_directo["Indice_Original"].duplicated().sum()

np.int64(56969)

In [6]:
# ==========================================================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Preparar una segunda opción 2 para poder resumir los valores de UCI 
# Insumo:      Tabla de homologacion UCI
# Resultado:   Valores numéricos y strings de la tabla UCI
# ==========================================================================================
num_cols = uci.select_dtypes(include="number").columns.tolist()
cat_cols = uci.select_dtypes(exclude="number").columns.tolist()

print("Numéricas :", num_cols)
print("Categóricas:", cat_cols)

Numéricas : ['Precio', 'MPG_Carretera', 'MPG_Ciudad', 'RPM_Maximas', 'Caballos_Fuerza', 'Relacion_Compresion', 'Carrera_Piston', 'Diametro_Cilindro', 'Tamano_Motor', 'Num_Cilindros', 'Peso_Vacio', 'Altura', 'Ancho', 'Longitud', 'Distancia_Ejes', 'Num_Puertas', 'Perdidas_Normalizadas', 'Nivel_Riesgo']
Categóricas: ['Sistema_Combustible', 'Tipo_Motor', 'Ubicacion_Motor', 'Traccion', 'Carroceria', 'Aspiracion', 'Tipo_Combustible', 'Fabricante', 'Marca', 'Marca_Homologada']


In [7]:
# ==========================================================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Quitar aquellas columnas que no son necesarian para el analisis
# Insumo:      catalogos de las columnas (cat_cols)
# Resultado:   Eliminación de valores
# ==========================================================================================
cat_cols= [c for c in cat_cols if c not in ["Fabricante", "Marca", "Marca_Homologada"]]
print(cat_cols)

['Sistema_Combustible', 'Tipo_Motor', 'Ubicacion_Motor', 'Traccion', 'Carroceria', 'Aspiracion', 'Tipo_Combustible']


In [8]:
# ===========================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Resumen de UCI a una fila por fabricante (promedio en
#              numéricas, moda en categóricas) para que la integración
#              sea muchos a uno y no genere duplicados.
# Insumo:      uci, num_cols, cat_cols
# Resultado:   DataFrame uci_resumen.
# ===========================================================
# Diccionario de agregación: qué función aplicar a cada columna
reglas = {col: "mean" for col in num_cols}
reglas.update({col: (lambda s: s.mode().iloc[0]) for col in cat_cols })

uci_resumen = uci.groupby("Marca_Homologada", as_index=False).agg(reglas)

print("UCI original :", uci.shape)
print("UCI resumido :", uci_resumen.shape)
uci_resumen.head()

UCI original : (205, 28)
UCI resumido : (22, 26)


,Marca_Homologada,Precio,MPG_Carretera,MPG_Ciudad,RPM_Maximas,Caballos_Fuerza,Relacion_Compresion,Carrera_Piston,Diametro_Cilindro,Tamano_Motor,Num_Cilindros,Peso_Vacio,Altura,Ancho,Longitud,Distancia_Ejes,Num_Puertas,Perdidas_Normalizadas,Nivel_Riesgo,Sistema_Combustible,Tipo_Motor,Ubicacion_Motor,Traccion,Carroceria,Aspiracion,Tipo_Combustible
0,alfa romeo,15498.333333,26.666667,20.333333,5000.000000,125.333333,9.000000,2.943333,3.206667,137.333333,4.666667,2639.666667,50.000000,64.566667,169.600000,90.566667,2.000000,NaN,2.333333,mpfi,dohc,front,rwd,convertible,std,gas
1,audi,17859.166667,24.142857,18.857143,5500.000000,121.000000,8.400000,3.400000,3.172857,130.714286,4.857143,2800.714286,54.428571,68.714286,183.828571,102.271429,3.428571,161.000000,1.285714,mpfi,ohc,front,fwd,sedan,std,gas
2,bmw,26118.750000,25.375000,19.375000,5068.750000,138.875000,8.575000,3.167500,3.473750,166.875000,5.500000,2929.375000,54.825000,66.475000,184.500000,103.162500,3.250000,190.000000,0.375000,mpfi,ohc,front,rwd,sedan,std,gas
3,chevrolet,6007.000000,46.333333,41.000000,5300.000000,62.666667,9.566667,3.083333,2.990000,80.333333,3.666667,1757.000000,52.400000,62.500000,151.933333,92.466667,2.666667,100.000000,1.000000,2bbl,ohc,front,fwd,hatchback,std,gas
4,dodge,7875.444444,34.111111,28.000000,5388.888889,86.333333,8.634444,3.365556,3.094444,102.666667,4.000000,2151.333333,51.644444,64.166667,160.988889,95.011111,3.000000,133.444444,1.000000,2bbl,ohc,front,fwd,hatchback,std,gas


In [9]:
# ===========================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Integración de CarDekho con UCI RESUMIDO usando
#              Marca_Homologada como clave
# Insumo:      cd, uci_resumido
# Resultado:   DataFrame integrado_resumen.
# ===========================================================
integrado_resumen = pd.merge(cd, uci_resumen, on="Marca_Homologada", how="left", suffixes=("", "_UCI"))

print("Filas antes  (CarDekho):", cd.shape[0])
print("Filas después (join)   :", integrado_resumen.shape[0])

Filas antes  (CarDekho): 15411
Filas después (join)   : 15411


In [10]:
# ==========================================================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Comprobar si hay duplicados en esta nuevo dataset
# Insumo:      integrado_resumen
# Resultado:   Valor numércio sobre las repeticiones (se espera que sea 0)
# ==========================================================================================
integrado_resumen["Indice_Original"].duplicated().sum()

np.int64(0)

### Justificación

- Operación utilizada: se aplicó un left join (pd.merge con how="left"), ya que CarDekho es el dataset principal y la práctica solicita explícitamente conservar todas sus observaciones. Esta operación mantiene todos los registros de la tabla principal, incluidos los vehículos cuya marca no existe en UCI Automobile; en esos casos, las columnas provenientes de UCI quedan con valores nulos (NaN).
- Resumen previo de UCI: antes de la integración, UCI Automobile se resumió a un solo registro por fabricante mediante estadística descriptiva (promedio para las dimensiones cuantitativas y moda para las cualitativas). Esto fue necesario porque Marca_Homologada no es única en UCI: un join directo generaba una relación muchos a muchos con 56,969 registros duplicados. Con el resumen, la relación pasa a ser muchos a uno y el dataset integrado conserva los 15,411 registros originales de CarDekho, sin duplicados.
- Limitación: la integración es a nivel de fabricante y no de vehículo individual, ya que UCI Automobile no contiene una dimensión equivalente al modelo del auto que permita identificar cada vehículo de forma específica.

# B.10 Evaluación de la integración

In [11]:
# =================================================================================================
# La observaciones antes y después del join, se realizacon como verificación en la paso anterior
# Donde se obtuvieron 2 resultados. Primer resultado: El join sin resumir la tabla de UCI:
# Filas antes  (CarDekho): 15411 / Filas después (join): 72380
# Segundo resultado: El join con la tabla UCI resumida:
# Filas antes  (CarDekho): 15411 / Filas después (join): 15411
# =================================================================================================

In [12]:
# =====================================================================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Saber cuántos valores distintos hay en la columna Marca_Homologada
# Insumo:      integracion_resumen
# Resultado:   Valor numércio sobre las distintas "fabricantes" (marca) en nuestra tabla "integracion_resumen"
# ======================================================================================================
integrado_resumen["Marca_Homologada"].nunique()

30

In [13]:
# =====================================================================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Encontrar la correspondencia con el "fabricante" (marca)
# Insumo:      integracion_resumen
# Resultado:   Valor numércio sobre las correspondias encontradas
# ======================================================================================================
integrado_resumen[ integrado_resumen["Num_Cilindros"].notna()]["Marca_Homologada"].nunique()

12

In [15]:
# ===========================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Métricas de la integración: observaciones antes/después,
#              fabricantes con y sin correspondencia y porcentaje.
# Insumo:      cd, integrado_resumen, integrado_directo
# Resultado:   Resumen impreso de las métricas del inciso 10.
# ===========================================================
total_fabricantes = integrado_resumen["Marca_Homologada"].nunique()
con_corresp = integrado_resumen[ integrado_resumen["Num_Cilindros"].notna()]["Marca_Homologada"].nunique()
sin_corresp = total_fabricantes - con_corresp
porcentaje = con_corresp * 100 / total_fabricantes

print("Observaciones antes       :", cd.shape[0])
print("Observaciones después     :", integrado_resumen.shape[0])
print("Fabricantes en CarDekho   :", total_fabricantes)
print("Con correspondencia en UCI:", con_corresp)
print("Sin correspondencia       :", sin_corresp)
print(f"Porcentaje de correspondencia: {porcentaje:.1f}%")
print("Duplicados join directo   :", integrado_directo["Indice_Original"].duplicated().sum())
print("Duplicados join resumido  :", integrado_resumen["Indice_Original"].duplicated().sum())

Observaciones antes       : 15411
Observaciones después     : 15411
Fabricantes en CarDekho   : 30
Con correspondencia en UCI: 12
Sin correspondencia       : 18
Porcentaje de correspondencia: 40.0%
Duplicados join directo   : 56969
Duplicados join resumido  : 0


### Conclusiones de la integración

- Observaciones y duplicados: el resumen previo de UCI Automobile, con el promedio de las dimensiones cuantitativas y la moda de las cualitativas por fabricante, permitió que la integración conservara las 15,411 observaciones de CarDekho antes y después del join. El join directo, en cambio, generaba 56,969 registros duplicados, debido a que Marca_Homologada se repite en UCI (cada fabricante tiene varios modelos). Con el resumen se obtuvieron cero duplicados.
- Correspondencia: solo 12 de los 30 fabricantes de CarDekho (40%) encontraron correspondencia en UCI; los 18 restantes (60%) no la tienen. Esto se debe a la diferencia de época y de región entre ambos datasets: CarDekho contiene autos usados del mercado de la India (alrededor de 2021), mientras que UCI Automobile contiene autos de 1985, principalmente del mercado estadounidense. Por ello, muchos fabricantes de CarDekho, como Maruti, Tata o Mahindra, no aparecen en UCI.
- Correspondencia a nivel de registros: aunque el 40% de los fabricantes tiene correspondencia, solo el 29.4% de los autos de CarDekho la tiene. El porcentaje es menor porque las marcas con más registros en CarDekho, como Maruti y Hyundai, no tienen pareja en UCI, por lo que una gran parte de los autos queda sin información complementaria.

In [16]:
# ===========================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Exportación del dataset integrado para las siguientes
#              secciones de la práctica (C, D, E, F y G).
# Insumo:      integrado_resumen
# Resultado:   data/processed/cardekho_integrado.csv
# ===========================================================
os.makedirs("data/processed", exist_ok=True)
integrado_resumen.to_csv("data/processed/cardekho_integrado.csv", index=False, encoding="utf-8-sig")
print("Exportado:", integrado_resumen.shape)

Exportado: (15411, 42)


## C. Filtrado y selección de datos

In [17]:
# ===========================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Función que aplica un filtro y reporta las métricas del
#              inciso 14 (observaciones antes/después y porcentajes).
# Insumo:      DataFrame y condición booleana
# Resultado:   DataFrame filtrado y métricas impresas.
# ===========================================================
def reportar_filtro(df, condicion, descripcion):
    """Aplica 'condicion' sobre 'df' e imprime las métricas del filtro."""
    filtrado = df[condicion]
    antes = len(df)
    despues = len(filtrado)
    conservado = despues / antes * 100
    print(f"Filtro: {descripcion}")
    print(f"  Antes: {antes} | Después: {despues}")
    print(f"  Conservadas: {conservado:.1f}% | Eliminadas: {100 - conservado:.1f}%\n")
    return filtrado

### C.11 Filtros que consideran condiciones sobre dimensiones cuantitativas y cualitativas

In [21]:
# ===========================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Filtro 1: Sobre dimensiones cuantitativas y cualitativas
#              inciso 14 (observaciones antes/después y porcentajes).
# Insumo:      DataFrame y condición booleana
# Resultado:   DataFrame filtrado y métricas impresas.
# ===========================================================
f1 = reportar_filtro(integrado_resumen, integrado_resumen["Año"] > 2015, "Autos fabricados después de 2015")
f2 = reportar_filtro(integrado_resumen, integrado_resumen["Precio_Venta"] > 556000., "Autos con precio de venta mayor a 556,000.")
f3 = reportar_filtro(integrado_resumen, integrado_resumen["Kilómetros"] < 50000, "Autos con kilometraje menor a 50,000")
f4 = reportar_filtro(integrado_resumen, integrado_resumen["Combustible"] == "Petrol", "Autos con combustible Petrol")

Filtro: Autos fabricados después de 2015
  Antes: 15411 | Después: 7666
  Conservadas: 49.7% | Eliminadas: 50.3%

Filtro: Autos con precio de venta mayor a 556,000.
  Antes: 15411 | Después: 7705
  Conservadas: 50.0% | Eliminadas: 50.0%

Filtro: Autos con kilometraje menor a 50,000
  Antes: 15411 | Después: 7391
  Conservadas: 48.0% | Eliminadas: 52.0%

Filtro: Autos con combustible Petrol
  Antes: 15411 | Después: 7643
  Conservadas: 49.6% | Eliminadas: 50.4%



In [22]:
# ===========================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Filtros con condiciones combinadas mediante AND, OR y NOT
#              (inciso 12).
# Insumo:      integrado_resumen, reportar_filtro
# Resultado:   DataFrames f5, f6 y f7 con sus métricas.
# ===========================================================
# AND (&): deben cumplirse las tres condiciones al mismo tiempo
f5 = reportar_filtro(
    integrado_resumen,
    (integrado_resumen["Año"] > 2015) & (integrado_resumen["Transmisión"] == "Automatic") & (integrado_resumen["Precio_Venta"] < 1000000),
    "AND: fabricados después de 2015, automáticos y con precio menor a 1,000,000"
)

# OR (|): basta con que se cumpla al menos una de las dos condiciones
f6 = reportar_filtro(
    integrado_resumen,
    (integrado_resumen["Año"] >= 2020) | (integrado_resumen["Precio_Venta"] > 2000000),
    "OR: fabricados en 2020 o después, o con precio mayor a 2,000,000"
)

# NOT (~): invierte la condición; se quedan los autos que NO son Maruti
f7 = reportar_filtro(
    integrado_resumen,
    ~(integrado_resumen["Marca_Homologada"] == "maruti"),
    "NOT: autos que no son de la marca Maruti"
)

Filtro: AND: fabricados después de 2015, automáticos y con precio menor a 1,000,000
  Antes: 15411 | Después: 893
  Conservadas: 5.8% | Eliminadas: 94.2%

Filtro: OR: fabricados en 2020 o después, o con precio mayor a 2,000,000
  Antes: 15411 | Después: 995
  Conservadas: 6.5% | Eliminadas: 93.5%

Filtro: NOT: autos que no son de la marca Maruti
  Antes: 15411 | Después: 10419
  Conservadas: 67.6% | Eliminadas: 32.4%



In [23]:
# ===========================================================
# Autor:       Magaly
# Fecha:       04 octubre 2026
# Descripción: Filtros sobre tres dimensiones categóricas usando isin()
#              para seleccionar categorías específicas (inciso 13).
# Insumo:      integrado_resumen, reportar_filtro
# Resultado:   DataFrames f8, f9 y f10 con sus métricas.
# ===========================================================
# isin() conserva las filas cuyo valor está dentro de la lista indicada

# Combustible + Transmisión + Tipo_Vendedor
f8 = reportar_filtro(
    integrado_resumen,
    (integrado_resumen["Combustible"].isin(["Diesel", "CNG"]))
    & (integrado_resumen["Transmisión"] == "Manual")
    & (integrado_resumen["Tipo_Vendedor"].isin(["Dealer", "Trustmark Dealer"])),
    "Diesel o CNG, transmisión manual, vendidos por Dealer o Trustmark Dealer"
)

# Marca + Combustible + Tipo_Vendedor
f9 = reportar_filtro(
    integrado_resumen,
    (integrado_resumen["Marca_Homologada"].isin(["toyota", "honda", "hyundai"]))
    & (integrado_resumen["Combustible"] == "Petrol")
    & (integrado_resumen["Tipo_Vendedor"] == "Individual"),
    "Toyota, Honda o Hyundai, combustible Petrol, vendidos por particulares"
)

# Marca + Transmisión + Combustible
f10 = reportar_filtro(
    integrado_resumen,
    (integrado_resumen["Marca_Homologada"].isin(["bmw", "audi", "mercedes benz"]))
    & (integrado_resumen["Transmisión"] == "Automatic")
    & (integrado_resumen["Combustible"] == "Diesel"),
    "Marcas de lujo (BMW, Audi, Mercedes-Benz), automáticos y Diesel"
)

Filtro: Diesel o CNG, transmisión manual, vendidos por Dealer o Trustmark Dealer
  Antes: 15411 | Después: 3536
  Conservadas: 22.9% | Eliminadas: 77.1%

Filtro: Toyota, Honda o Hyundai, combustible Petrol, vendidos por particulares
  Antes: 15411 | Después: 971
  Conservadas: 6.3% | Eliminadas: 93.7%

Filtro: Marcas de lujo (BMW, Audi, Mercedes-Benz), automáticos y Diesel
  Antes: 15411 | Después: 812
  Conservadas: 5.3% | Eliminadas: 94.7%

